# MCORR AND CNMF
### Analysis of riboL1-GCaMP8m responses imaged with 16x objective at zoom 1x over 765x765 pixels

## Import

In [ ]:
import os
import glob
from copy import deepcopy

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import mesmerize_core as mc
import mesmerize_viz
import fastplotlib as fpl

import bruker_concat_tif as ct
from tifffile import TiffWriter

from mesmerize_core.caiman_extensions.cnmf import cnmf_cache

from pathlib import Path
from PIL import Image

if os.name == "nt":
    # disable the cache on windows, this will be automatic in a future version
    cnmf_cache.set_maxsize(0)

pd.options.display.max_colwidth = 120

## Set paths

In [ ]:
# data path
data_path = Path('D:/Data/2P/C57_O1M2/10022023/TSeries-10022023-1300-001')
print(f"Load data from {data_path}.")

# output path
export_path = Path('H:/Analysis/2P/C57_O1M2/10022023/run1/mesmerize/')

# Create export_path if it doesn't exist
if not os.path.exists(export_path):
    os.makedirs(export_path)
    
mc.set_parent_raw_data_path(export_path) 

# movie path
movie_path = Path.joinpath(export_path, 'cat_tiff_bt.tiff')

# batch path
batch_path = Path.joinpath(export_path, 'batch.pickle')

print(f"Export data to {export_path}.") 
print(f"Movie path: {movie_path}.") 
print(f"Batch path: {batch_path}.") 


### Concatenate the ome.tif files into a single multi-page tiff file

In [ ]:
# Using the concat_tif.py script to concatenate the tif files. If needed, install libtiff with: pip install pylibtiff
ct.concatenate_files([data_path], export_path)

### Create a new batch

This creates a new pandas `DataFrame` with the columns that are necessary for mesmerize. In mesmerize we call this the **batch DataFrame**. You can add additional columns relevant to your experiment, but do not modify columns used by mesmerize.

Note that when you create a DataFrame you will need to use `load_batch()` to load it later. You cannot use `create_batch()` to overwrite an existing batch DataFrame

In [ ]:
# create a new batch
df = mc.create_batch(batch_path)
# to load existing batches use `load_batch()`
# df = mc.load_batch(batch_path)
df

### Define parameters and add to batch, algo = mcorr


In [ ]:
# copy the mcorr_params2 dict to make some changes
# new_params = deepcopy(mcorr_params2)
default_params = \
{
  'main':
    {
        'strides': [36, 36],
        'overlaps': [24, 24],
        'max_shifts': [12, 12],
        'max_deviation_rigid': 6,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

# new_params_1 = \
# {
#   'main':
#     {
#         'strides': [18, 18],
#         'overlaps': [24, 24],
#         'max_shifts': [12, 12],
#         'max_deviation_rigid': 6,
#         'border_nan': 'copy',
#         'pw_rigid': True,
#         'gSig_filt': None
#     },
# }

df.caiman.add_item(
  algo='mcorr',
  item_name=movie_path.stem,
  input_movie_path=movie_path,
  params=default_params
)
# df.caiman.add_item(
#   algo='mcorr',
#   item_name=movie_path.stem,
#   input_movie_path=movie_path,
#   params=new_params_1
# )

df


### Run batch items.

`df.iterrows()` iterates through rows and returns the numerical index and row for each iteration

In [ ]:
for i, row in df.iterrows():
    # If already processed, skip
    if row["outputs"] is not None:
        print(f"Skipping batch item {i}, id {row.uuid}, algo {row.algo}. Already run.") 
        continue
    process = row.caiman.run()
    
    # on Windows you MUST reload the batch dataframe after every iteration because it uses the `local` backend.
    # this is unnecessary on Linux & Mac
    # "DummyProcess" is used for local backend so this is automatic
    if process.__class__.__name__ == "DummyProcess":
        df = df.caiman.reload_from_disk()

### Load outputs to dataframe

In [ ]:
df = df.caiman.reload_from_disk()
df

### Visualize using fastplotlib


In [ ]:
# first item is just the raw movie
movies = [df.iloc[0].caiman.get_input_movie()]

# subplot titles
subplot_names = ["raw"]

# add all the mcorr outputs to the list
for i, row in df.iterrows():
    # Select which row to display
    # if i == 2 or i == 3:
    # add to the list of movies to plot
    movies.append(row.mcorr.get_output())

    # subplot title to show dataframe index
    subplot_names.append(f"ix {i}")

# create the widget
mcorr_iw_multiple = fpl.ImageWidget(
    data=movies,  # list of movies
    window_funcs={"t": (np.mean, 1)}, # window functions as a kwarg, this is what the slider was used for in the ready-made viz
    grid_plot_kwargs={"size": (900, 900)},
    names=subplot_names,  # subplot names used for titles
    cmap="gray" #"gnuplot2"
)

mcorr_iw_multiple.show()

In [ ]:
mcorr_iw_multiple.close()

#### Clip and save the motion corrected memmap values to the 0-65535 range

In [ ]:
# Get the motion corrected output as a memmaped numpy array
mcorr_movie = df.iloc[0].mcorr.get_output() 
# Clip, but don't convert to uint16 yet (caiman expects 32 bit float)
mcorr_movie_16bit = np.clip(mcorr_movie, 0, 2**16-1)

# get path of mcorr obj on disk so we can overwrite
mcorr_path = df.iloc[0].mcorr.get_output_path()
original_mmap_path = Path(mcorr_path)
print(f"mcorr_movie path: {mcorr_path}")

# Create a new memmap file with write access
clipped_mmap_path = original_mmap_path.parent / f"clipped_{original_mmap_path.name}"
clipped_mcorr_movie = np.memmap(clipped_mmap_path, dtype='float32', mode='w+', shape=mcorr_movie_16bit.shape)

# Copy the data from mcorr_movie_16bit to the new memmap file
np.copyto(clipped_mcorr_movie, mcorr_movie_16bit)

# Flush changes to disk and close the memmap and the original memmap
del clipped_mcorr_movie
del mcorr_movie

# Code below does not work on Windows while memmap is open

# Create a backup copy of the original memmap file then delete the original
# original_mmap_path.rename(original_mmap_path.parent / f"original_{original_mmap_path.name}")
# print(f"Original motion corrected movie saved to {original_mmap_path.parent / f'original_{original_mmap_path.name}'}")

# Finally, rename the clipped memmap file to the original name 
# clipped_mmap_path.rename(original_mmap_path)
# Delete the backup copy 
# (original_mmap_path.parent / f"original_{original_mmap_path.name}").unlink()

#### Save the motion corrected movie (memmaped array) as a BigTIFF

In [ ]:
# Convert values to uint8, and save a 8 bit version
mcorr_tif_path = Path.joinpath(export_path, f"mcorr_u8.tiff")
mcorr_movie_8bit = (mcorr_movie_16bit / (2**16-1) * 255).astype('uint8')
with TiffWriter(mcorr_tif_path, bigtiff=True) as tif:
    tif.write(mcorr_movie_8bit, photometric='minisblack')
   
print(f"Saved 8 bit motion corrected movie to {mcorr_tif_path}.")

#### Create an excerpt of the original movie vs motion corrected movie

In [ ]:
# We take the 80 first frames of the original movie and the motion corrected movie, concatenate them horizontally and save the resulting array as a mp4 movie

original_movie = df.iloc[0].caiman.get_input_movie()

# Keep only the first 80 frames of the movies 
original_movie_excerpt = original_movie[:80]
mcorr_movie_excerpt = mcorr_movie_16bit[:80]

### for visualization purposes only ###
# If ranges are significantly different, consider rescaling
original_min, original_max = np.min(original_movie_excerpt), np.max(original_movie_excerpt)
mcorr_min, mcorr_max = np.min(mcorr_movie_excerpt), np.max(mcorr_movie_excerpt)
if original_min != mcorr_min or original_max != mcorr_max:
    normalized_original = (original_movie_excerpt - original_min) / (original_max - original_min) * 65535
    normalized_mcorr = (mcorr_movie_excerpt - mcorr_min) / (mcorr_max - mcorr_min) * 65535
else:
    normalized_original = original_movie_excerpt
    normalized_mcorr = mcorr_movie_excerpt

# Convert to uint16 after normalization
normalized_original_uint16 = normalized_original.astype(np.uint16)
normalized_mcorr_uint16 = normalized_mcorr.astype(np.uint16)

# Concatenate the two movies horizontally
excerpt_movie = np.concatenate((normalized_original_uint16, normalized_mcorr_uint16), axis=2)

# Set the path of the mp4 movie
excerpt_movie_mp4_path = Path.joinpath(export_path, f"excerpt_movie.mp4")
# # Display the first frame of the excerpt_movie 
# first_frame = excerpt_movie[0]
# plt.imshow(first_frame, cmap='gray')
# plt.show()

# Create a directory for temporary frame storage
temp_frame_dir = Path(export_path) / "temp_frames"
os.makedirs(temp_frame_dir, exist_ok=True)

# Save each frame as an image with added padding, to avoid [libx264 @ 0x563ae05f8b40] height not divisible by 2 error
for i, frame in enumerate(excerpt_movie):
    # Add a row of black pixels at the bottom
    padded_frame = np.pad(frame, ((0, 1), (0, 0)), 'constant')
    frame_image = Image.fromarray(padded_frame)
    frame_image.save(temp_frame_dir / f"frame_{i:04d}.png")

# Construct the FFmpeg command to make an MP4 movie with the adjusted size
ffmpeg_command = f"ffmpeg -y -r 30 -f image2 -s 1530x766 -i {temp_frame_dir}/frame_%04d.png -vcodec libx264 -crf 25 -pix_fmt yuv420p {export_path}/excerpt_movie.mp4"
os.system(ffmpeg_command)

# Delete the temporary frame images after creating the video
for file in os.listdir(temp_frame_dir):
    os.remove(temp_frame_dir / file)
os.rmdir(temp_frame_dir)

print(f"Saved excerpt movie to {export_path}/excerpt_movie.mp4")

## CNMF 

### Replace mmap
If a clipped memmap file exists, delete the original memmap file and rename the clipped memmap file to the original name

In [ ]:

outputs_dict = df.iloc[0]["outputs"]
mc_memmapped_fname = Path.joinpath(export_path, outputs_dict['mcorr-output-path'])
print(f"Original memmap file: {mc_memmapped_fname}.")

# Check if a clipped memmap file exists (i.e. same name as original but with "clipped_" appended to the beginning)
clipped_mmap_path = Path.joinpath(mc_memmapped_fname.parent,
                                  f"clipped_{mc_memmapped_fname.name}")
# check if the clipped memmap file exists
if clipped_mmap_path.exists():
    print(f"Clipped memmap file exists: {clipped_mmap_path}.")
    # if it does, delete the original memmap file
    mc_memmapped_fname.unlink()
    # and rename the clipped memmap file to the original name
    clipped_mmap_path.rename(mc_memmapped_fname)
    print(f"Clipped memmap file renamed to {mc_memmapped_fname}.")

### CNMF
Perform CNMF using the mcorr output.
Similar to mcorr, put the CNMF params within the `main` key. The `refit` key will perform a second iteration, as shown in the `caiman` `demo_pipeline.ipynb` notebook.

### Set params

In [ ]:
# some params for CNMF
params_cnmf =\
{
    'main': # indicates that these are the "main" params for the CNMF algo
        {
            'fr': 20, # framerate, very important!
            'p': 1,
            'nb': 2,
            'merge_thr': 0.85,
            'rf': 20,
            'stride': 10, # "stride" for cnmf, "strides" for mcorr
            'K': 10,
            'gSig': [5, 5],
            'ssub': 1,
            'tsub': 1,
            'method_init': 'greedy_roi',
            'min_SNR': 3.0,
            'SNR_lowest':  1.0,
            'rval_thr': 0.8,
            'rval_lowest': 0.2,
            'use_cnn': True,
            'min_cnn_thr': 0.9,
            'cnn_lowest': 0.2,
            'decay_time': 0.15,
        },
    'refit': True, # If `True`, run a second iteration of CNMF
}
# Create second set of parameters
# new_params_cnmf = deepcopy(params_cnmf)
# new_params_cnmf["main"]["gSig"] = [4, 4]

### Add CNMF item

You can provide the mcorr item row to `input_movie_path` and it will resolve the path of the input movie from the entry in the DataFrame.

In [ ]:
good_mcorr_index = 0
 
# add batch items
df.caiman.add_item(
    algo='cnmf', # algo is cnmf
    input_movie_path=df.iloc[good_mcorr_index],  # use mcorr output from a completed batch item
    params=params_cnmf,
    item_name=df.iloc[good_mcorr_index]["item_name"], # use the same item name
)
# add a batch item
# df.caiman.add_item(
#     algo='cnmf', # algo is cnmf
#     input_movie_path=df.iloc[good_mcorr_index],  # use mcorr output from a completed batch item
#     params=new_params_cnmf,
#     item_name=df.iloc[good_mcorr_index]["item_name"], # use the same item name
# )

See the cnmf item at the bottom of the dataframe

In [ ]:
df

### Run CNMF

In [ ]:
# Run rows that haven't been run yet
for i, row in df.iterrows():
    if row["outputs"] is not None: # item has already been run
        continue # skip
        
    process = row.caiman.run()
    
    # on Windows you MUST reload the batch dataframe after every iteration because it uses the `local` backend.
    # this is unnecessary on Linux & Mac
    # "DummyProcess" is used for local backend so this is automatic
    if process.__class__.__name__ == "DummyProcess":
        df = df.caiman.reload_from_disk()

### Save parameters to disk as a json file

In [ ]:
import json
params_path = Path.joinpath(export_path, 'caiman_params.json')

caiman_params = {
    'data_path': str(data_path),
    'export_path': str(export_path),
    'movie_name': str(movie_path.name)
}
caiman_mcorr_params = df.iloc[0]["params"]['main']
caiman_mcorr_params['timestamp_mcorr'] = df.iloc[0]["ran_time"]
caiman_cnmf_params = params_cnmf['main'] | df.iloc[-1]["params"]['main']
caiman_cnmf_params['timestamp_cnmf'] = df.iloc[-1]["ran_time"]
caiman_params = caiman_params | caiman_mcorr_params | caiman_cnmf_params

with open(params_path, 'w') as f:
    json.dump(caiman_params, f, indent=4) #sort_keys=True)

### Save to Matlab

In [ ]:
# View the content of estimates
# cnmf_obj.estimates?

In [ ]:
# Prepare outputs for saving
cnmf_obj = df.iloc[-1].cnmf.get_output()

# Keep only accepted components
cnmf_obj.estimates.select_components(use_object=True)

# Compute dF/F
if cnmf_obj.estimates.F_dff is None:
    print('Calculating estimates.F_dff')
    cnmf_obj.estimates.detrend_df_f(quantileMin=8, 
                                      frames_window=400,
                                      use_residuals=False);  # use denoised data

In [ ]:
# save results for matlab
from scipy import io
# Lists must be converted to arrays, otherwise Matlab loads them as chars
io.savemat(Path.joinpath(export_path, 'results_caiman.mat'), mdict={
                                                 'mean_map_motion_corrected': df.iloc[0].caiman.get_projection("mean"),
                                                 'spatial_components': cnmf_obj.estimates.A,
                                                 'temporal_components': cnmf_obj.estimates.C,
                                                 'background_spatial_component': cnmf_obj.estimates.b,
                                                 'background_temporal_component': cnmf_obj.estimates.f,
                                                 'df_wo_bckgrnd': cnmf_obj.estimates.F_dff,
                                                 'deconv_spk': cnmf_obj.estimates.S,
                                                 'SNR_comp': cnmf_obj.estimates.SNR_comp,
                                                 'baseline': np.array(cnmf_obj.estimates.bl, dtype=np.float32),
                                                 'noise': np.array(cnmf_obj.estimates.neurons_sn, dtype=np.float32)})